# Generative LLM — `career_position` Annotation

**Model:** Qwen3 80B (LM Studio)  
**Granularity:** Broad sector codes  
**Prompting:** zero-shot (set `N_SHOTS > 0` for few-shot)  

> LM Studio must be running at `LMSTUDIO_URL` before executing inference.

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import re
import time
from tqdm.auto import tqdm
from openai import OpenAI

from corex_eval import evaluate, load_inputs, load_training_data, career_position_to_sector

# --- Config ---
LMSTUDIO_URL = "http://localhost:1234/v1"
MODEL_ID     = "qwen2.5-72b-instruct"   # ← verify with the "Discover models" cell below
N_SHOTS      = 0    # 0 = zero-shot; set to 3 or 5 for few-shot
SEED         = 42

## 2. Set up client & discover model IDs

In [ ]:
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")

print("Models available in LM Studio:")
try:
    for m in client.models.list():
        print(f"  {m.id}")
except Exception as e:
    print(f"  Could not connect to LM Studio at {LMSTUDIO_URL}: {e}")

## 3. Load valid codes

In [ ]:
from corex_eval.config import CAREER_POSITION_SECTORS, CAREER_POSITION_SECTOR_HINTS

train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position"],
)
train_df = train_df.dropna(subset=["job_description_label", "career_position"]).reset_index(drop=True)
train_df["sector"] = train_df["career_position"].map(career_position_to_sector)

valid_codes = sorted(CAREER_POSITION_SECTORS.keys())
sector_lines = [
    f"  {k} = {v}: {CAREER_POSITION_SECTOR_HINTS[k]}"
    for k, v in sorted(CAREER_POSITION_SECTORS.items())
]
print(f"{len(valid_codes)} broad sector codes:")
for line in sector_lines:
    print(line)

## 4. Build prompt & prediction parser

In [ ]:
def build_system_prompt(sector_lines):
    codes_str = "\n".join(sector_lines)
    return (
        "You are an expert political scientist coding political career histories "
        "using the CoREx codebook.\n\n"
        "Task: given a career entry (job title and workplace), assign the single most "
        "appropriate broad sector code.\n"
        'Respond with ONLY the sector number (e.g. "1") — nothing else.\n\n'
        f"Broad sector codes:\n{codes_str}"
    )

def build_messages(job_description, workplace, system_prompt, shot_examples):
    user_content = f"Job title: {job_description}\nWorkplace: {workplace or 'unknown'}"
    messages = [{"role": "system", "content": system_prompt}]
    for desc, wp, sector in shot_examples:
        messages.append({"role": "user", "content": f"Job title: {desc}\nWorkplace: {wp or 'unknown'}"})
        messages.append({"role": "assistant", "content": sector})
    messages.append({"role": "user", "content": user_content})
    return messages

def parse_prediction(response_text, valid_codes):
    text = response_text.strip().strip("\"'")
    if text in valid_codes:
        return text
    import re
    match = re.search(r"\b(\d+)\b", text)
    if match and match.group(1) in valid_codes:
        return match.group(1)
    return text

SYSTEM_PROMPT = build_system_prompt(sector_lines)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 5. Few-shot examples (skipped when `N_SHOTS=0`)

In [ ]:
if N_SHOTS > 0:
    shot_pool = (
        train_df.groupby("sector", group_keys=False)
        .apply(lambda g: g.sample(1, random_state=SEED))
        .reset_index(drop=True)
    )
    shot_examples = (
        shot_pool.sample(min(N_SHOTS, len(shot_pool)), random_state=SEED)
        [["job_description_label", "workplace_label", "sector"]]
        .values.tolist()
    )
    print(f"Using {len(shot_examples)} few-shot examples")
else:
    shot_examples = []
    print("Zero-shot mode")

## 6. Load test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df.head()

## 7. Run inference

In [ ]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    messages = build_messages(
        row["job_description_label"],
        row.get("workplace_label", ""),
        SYSTEM_PROMPT,
        shot_examples,
    )
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=0,
            max_tokens=64,
            seed=SEED,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.1)
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 8. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="broad",
    submit=True,
    experiment_path="experiments/annotation/lmstudio_qwen3_80b_broad/config.yaml",
)